# IN403 – Unit 5 Lab: Embeddings + Vector Search + RAG (Hospital Policy Assistant)

**Goal:** Build a Retrieval-Augmented Generation (RAG) assistant that answers policy questions **only** from the provided hospital documents.

## What you will submit
1) Completed notebook (`.ipynb`) with outputs visible
2) A short deployment memo (≈ 300–500 words) answering:
   - Would you deploy this in a hospital? Under what guardrails?
   - What failure modes remain even with RAG?

## Files provided
- `IN403_Unit5_Hospital_Policy_Dataset.zip`

## Success criteria
- You can ask a question and see:
  - the **top retrieved policy chunks** (with document names)
  - a **grounded answer** that cites the retrieved chunks
  - a comparison against a **non-RAG** (plain LLM) answer


## 0) Setup
This lab uses:
- `sentence-transformers` for embeddings
- `faiss-cpu` for vector search
- `transformers` for text generation (small model)

If running in Google Colab, run the install cell below.


In [ ]:
# If running in Colab, uncomment installs:
# !pip -q install sentence-transformers faiss-cpu transformers accelerate

import os, re, zipfile, math
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer

from transformers import pipeline


## 1) Unzip dataset
Upload `IN403_Unit5_Hospital_Policy_Dataset.zip` to your runtime, then run this cell.


In [ ]:
zip_file = 'IN403_Unit5_Hospital_Policy_Dataset.zip'
data_dir = 'policy_data'

if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(data_dir)

sorted(os.listdir(data_dir))[:10]


## 2) Load policy documents

**Student task:** Load all `.txt` files and store:
- `doc_id`
- `doc_name`
- `text`


In [ ]:
def load_txt_documents(folder: str) -> List[Dict]:
    docs = []
    for fn in sorted(os.listdir(folder)):
        if fn.lower().endswith('.txt'):
            path = os.path.join(folder, fn)
            with open(path, 'r', encoding='utf-8') as f:
                text = f.read().strip()
            docs.append({'doc_id': len(docs), 'doc_name': fn, 'text': text})
    return docs

documents = load_txt_documents(data_dir)
print('Docs loaded:', len(documents))
documents[0]['doc_name']


## 3) Chunk documents

LLMs work better when you retrieve small chunks (e.g., 200–400 words) rather than whole documents.

**Requirements:**
- Chunk size: ~300 words
- Overlap: ~50 words
- Keep metadata: `doc_name`, `chunk_id`


In [ ]:
def normalize_whitespace(s: str) -> str:
    return re.sub(r'\s+', ' ', s).strip()

def chunk_text(text: str, chunk_words: int = 300, overlap_words: int = 50) -> List[str]:
    words = normalize_whitespace(text).split(' ')
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start = max(0, end - overlap_words)
    return chunks

chunk_rows = []
for d in documents:
    chunks = chunk_text(d['text'], chunk_words=300, overlap_words=50)
    for i, ch in enumerate(chunks):
        chunk_rows.append({
            'doc_name': d['doc_name'],
            'chunk_id': i,
            'text': ch
        })

chunks_df = pd.DataFrame(chunk_rows)
print('Chunks:', len(chunks_df))
chunks_df.head()


## 4) Create embeddings

We’ll use a small embedding model to convert each chunk into a vector.

**Model recommendation:** `sentence-transformers/all-MiniLM-L6-v2`


In [ ]:
embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = SentenceTransformer(embed_model_name)

chunk_texts = chunks_df['text'].tolist()
embeddings = embedder.encode(chunk_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype='float32')
embeddings.shape


## 5) Build a FAISS vector index

We’ll use cosine similarity. With normalized vectors, cosine similarity = inner product.


In [ ]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print('Index size:', index.ntotal)


## 6) Retrieval function

**Student task:** Given a question, retrieve top-k relevant chunks and display:
- similarity score
- doc_name
- chunk_id
- chunk text


In [ ]:
def retrieve(query: str, k: int = 3) -> pd.DataFrame:
    q_emb = embedder.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype='float32')
    scores, idxs = index.search(q_emb, k)
    rows = []
    for score, idx in zip(scores[0], idxs[0]):
        row = chunks_df.iloc[int(idx)].to_dict()
        row['score'] = float(score)
        rows.append(row)
    return pd.DataFrame(rows).sort_values('score', ascending=False)

test_query = 'What should staff do if they suspect a privacy incident?'
retrieve(test_query, k=3)


## 7) Build a grounded RAG answer

We will:
1) Retrieve top-k chunks
2) Create a prompt that instructs the model to answer ONLY from context
3) Generate an answer and include citations (doc_name + chunk_id)

### Important constraint
If the context does not contain the answer, the assistant must say **"I don't know based on the provided policies."**


In [ ]:
gen = pipeline('text2text-generation', model='google/flan-t5-small')

def build_context(retrieved_df: pd.DataFrame) -> str:
    blocks = []
    for _, r in retrieved_df.iterrows():
        blocks.append(f"[SOURCE: {r['doc_name']} | chunk {r['chunk_id']}]\n{r['text']}")
    return '\n\n'.join(blocks)

def rag_answer(question: str, k: int = 3, max_new_tokens: int = 220) -> Tuple[str, pd.DataFrame]:
    retrieved = retrieve(question, k=k)
    context = build_context(retrieved)
    prompt = (
        "You are a hospital policy assistant. Answer the QUESTION using only the POLICY CONTEXT. "
        "If the answer is not in the context, say: 'I don't know based on the provided policies.' "
        "Be concise. Include citations as (doc_name, chunk_id).\n\n"
        f"POLICY CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER:"
    )
    out = gen(prompt, max_new_tokens=max_new_tokens)[0]['generated_text']
    return out, retrieved

q = 'If a staff member suspects a privacy incident, what steps must they take and how fast?'
ans, retrieved = rag_answer(q, k=3)
print(ans)
retrieved


## 8) Compare against a non-RAG answer

A 'plain LLM' has no access to policies. This comparison shows hallucination risk.

**Student task:** Compare:
- Plain answer
- RAG answer
- Identify any claims in the plain answer that aren't grounded in policy


In [ ]:
plain_prompt = (
    "You are a helpful assistant. Answer the question as best you can.\n"
)

def plain_answer(question: str, max_new_tokens: int = 220) -> str:
    out = gen(plain_prompt + question, max_new_tokens=max_new_tokens)[0]['generated_text']
    return out

q = 'What PPE is required for aerosol-generating procedures?'
print('PLAIN ANSWER:\n', plain_answer(q))
print('\nRAG ANSWER:\n', rag_answer(q, k=3)[0])


## 9) Mini evaluation set

**Student task:** Run 5 questions and record whether the RAG answer is grounded.
Use this as a lightweight evaluation.


In [ ]:
eval_questions = [
    'How long should staff wash hands when hands are visibly soiled?'
  , 'What are the first actions during a Code Blue?'
  , 'What are the steps after a patient fall?'
  , 'What should staff do if barcode scanning fails?'
  , 'How quickly must a privacy incident be reported?'
]

rows = []
for q in eval_questions:
    rag, ret = rag_answer(q, k=3)
    rows.append({
        'question': q,
        'rag_answer': rag,
        'top_sources': '; '.join([f"{r.doc_name}:{int(r.chunk_id)}" for r in ret.itertuples()])
    })

pd.DataFrame(rows)


## 10) Reflection + Deployment Memo

Answer in markdown:
1) Where did retrieval help the most?
2) What are remaining risks (outdated policies, missing policies, chunking errors)?
3) What guardrails would you require (human review, logging, confidence thresholds, document governance)?
4) Would you deploy this? Why or why not?

### Optional AI Extension (bonus)
- Add a 'refusal' threshold: if the top similarity score is below a threshold, the model must refuse.
- Add source ranking and show the top similarity score.
